# Figure 2: interactive explorationFigure 2 is produced by `experiments/run_loss_figures.py`, which runs the full sweep andwrites the panels under their manuscript filenames. Use that for the reportedfigures.This notebook is for inspecting a single mean function and error distribution without running the whole sweep.

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

In [ ]:
import osMEAN_FUNCS = {"interaction": "nonlinear",              "gam": "n1000_gam",              "linear": "n1000_linear"}ERRORS = {"normal": "normal", "gumbel": "gumbel", "logistic": "log"}N_TRAIN, CENSORING, EPOCHS = 1000, 0.25, 30OUTDIR = "manuscript/images" if os.path.isdir("manuscript/images") else "results"os.makedirs(OUTDIR, exist_ok=True)print("writing figures to", OUTDIR)

In [ ]:
losses = {}for mf, mf_tag in MEAN_FUNCS.items():    for err, err_tag in ERRORS.items():        seeds = make_seeds(42)        rng = seeds.data()        tau = D.calibrate_tau(N_TRAIN, D.MEAN_FUNCTIONS[mf], err, rng,                              CENSORING, D.DEPENDENCE_SPECS["ar1"])        tr = D.make_dataset(N_TRAIN, mf, err, rng, dependence="ar1", tau=tau)        # The figure plots training loss only, so no test set is generated and        # no held-out evaluation is run; both would be discarded.        cfg = TrainConfig(model="rnn_agt", epochs=EPOCHS, pair_sample_s=30,                          hidden_dim=64, gru_layers=2, lr=3e-4)        res = train_model(tr, None, 3, cfg, make_seeds(11))        losses[(mf, err)] = res.epoch_losses        print(f"{mf:12s} {err:9s} final loss {res.epoch_losses[-1]:.4f}", flush=True)

In [ ]:
for (mf, err), curve in losses.items():    fname = f"loss_{ERRORS[err]}_{MEAN_FUNCS[mf]}.png"    fig, ax = plt.subplots(figsize=(4.2, 3.0))    ax.plot(range(1, len(curve) + 1), curve, lw=1.4, color="tab:blue")    ax.set_xlabel("epoch")    ax.set_ylabel("mini-batch Gehan-WRS loss")    ax.set_title(f"{mf}, {err}", fontsize=10)    ax.grid(alpha=.3)    fig.tight_layout()    fig.savefig(os.path.join(OUTDIR, fname), dpi=200)    plt.close(fig)    print("wrote", fname)print("\n9 panels written. The manuscript arranges them into fig:loss_all_layout;")print("panel order there is interaction (a-c), GAM (d-f), linear (g-i).")